# Gaia DR3 Variable Stars: Feature Engineering

## Imports and Configuration

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from astropy.timeseries import LombScargle
from astropy_healpix import HEALPix
from astropy import units as u
from IPython.display import display
from joblib import Parallel, delayed

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

for path in (root / "src", root):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from gaia_variable_stars.config import (
    SOURCE_FILE, CLEAN_FILE, PHOTOMETRY_FILE,
    FEATURE_FILE, CLASSES, MIN_POINTS, SEED
)

pd.set_option("display.max_columns", 100)
MIN_PERIOD = 0.1
MAX_PERIOD = 1000.0
GRID_SIZE = 5
MAX_DRIFT = 0.25
N_PEAKS = 5
N_HARMONICS = 4
CORR_LIMIT = 0.90
USE_WEIGHTS = False

FE_FEATURE_FILE = FEATURE_FILE.with_name("gaia_features_fe.csv")
FE_SAMPLING_FILE = FEATURE_FILE.with_name("gaia_features_fe_sampling.csv")
FE_VALIDATION_FILE = FEATURE_FILE.with_name("gaia_period_validation_fe.csv")
FE_ORACLE_FILE = FEATURE_FILE.with_name("gaia_features_fe_oracle.csv")
FE_SUBSAMPLE_FILE = FEATURE_FILE.with_name("gaia_fe_subsampling.csv")
FE_WRONG_FILE = FEATURE_FILE.with_name("gaia_fe_wrong_period.csv")
FE_RESULTS_FILE = FEATURE_FILE.with_name("gaia_fe_classification.csv")
FE_CORR_FILE = FEATURE_FILE.with_name("gaia_fe_selected.csv")

## Load Data

In [3]:
sources = pd.read_csv(SOURCE_FILE)
photometry = pd.read_csv(CLEAN_FILE, dtype={"transit_id": "string"})

source_cols = [
    "source_id", "class_name", "bp_rp_color", "ecl_lat",
    "ra", "dec", "g_mean_magnitude", "gaia_period",
    "gaia_period_origin"
]
photo_cols = [
    "source_id", "g_transit_time", "g_transit_mag",
    "g_transit_flux", "g_transit_flux_error", "variability_flag_g_reject"
]
missing = (set(source_cols) - set(sources)) | (set(photo_cols) - set(photometry))
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")
if sources["source_id"].duplicated().any():
    raise ValueError("Duplicate source IDs.")
if not sources["class_name"].isin(CLASSES).all():
    raise ValueError("Unexpected variable-star class.")
if not set(photometry["source_id"]).issubset(set(sources["source_id"])):
    raise ValueError("Photometry contains unknown source IDs.")

for col in ["bp_rp_color", "ecl_lat", "ra", "dec", "gaia_period", "g_mean_magnitude"]:
    sources[col] = pd.to_numeric(sources[col], errors="coerce")

print("Sources:", len(sources))
print("Clean observations:", len(photometry))
display(sources["class_name"].value_counts().reindex(CLASSES, fill_value=0))

Sources: 2000
Clean observations: 85243


class_name
RR Lyrae                500
Cepheid                 500
Eclipsing Binary        500
Long-Period Variable    500
Name: count, dtype: int64

## Prepare Photometry

In [4]:
observations = photometry.copy()
for col in ["g_transit_time", "g_transit_mag", "g_transit_flux", "g_transit_flux_error"]:
    observations[col] = pd.to_numeric(observations[col], errors="coerce")

valid = (
    np.isfinite(observations[["g_transit_time", "g_transit_mag", "g_transit_flux"]]).all(axis=1)
    & observations["g_transit_flux"].gt(0)
    & observations["variability_flag_g_reject"].astype("string")
      .str.strip().str.lower().isin(["false", "0"])
)
if not valid.all():
    raise ValueError("CLEAN_FILE contains invalid or rejected observations.")
if observations.duplicated(["source_id", "g_transit_time"]).any():
    raise ValueError("Duplicate source timestamps in CLEAN_FILE.")

flux = observations["g_transit_flux"]
error = observations["g_transit_flux_error"]
observations["magnitude_error"] = (
    2.5 / np.log(10) * error / flux
).where(np.isfinite(error) & error.gt(0))
observations = observations.sort_values(
    ["source_id", "g_transit_time"]
).reset_index(drop=True)

print("Valid observations:", len(observations))
print("Valid magnitude errors:", observations["magnitude_error"].notna().sum())

Valid observations: 85243
Valid magnitude errors: 85243


## Period Search and Fourier Fits

In [9]:
def period_search(time, mag, errors=None):
    baseline = np.ptp(time)
    max_period = min(MAX_PERIOD, baseline / 2)
    searchable = len(time) >= MIN_POINTS and np.unique(time).size >= MIN_POINTS and np.std(mag) > 0
    if not searchable or max_period <= MIN_PERIOD:
        return None

    weighted = USE_WEIGHTS and errors is not None and np.isfinite(errors).all() and (errors > 0).all()
    periodogram = LombScargle(time - time.min(), mag, dy=errors if weighted else None,
                              normalization="standard")
    frequency, powers = periodogram.autopower(
        minimum_frequency=1 / max_period, maximum_frequency=1 / MIN_PERIOD,
        samples_per_peak=GRID_SIZE,
    )
    valid = (frequency >= 1 / max_period) & (frequency <= 1 / MIN_PERIOD)
    frequency, powers = frequency[valid], powers[valid]
    finite = np.isfinite(powers)
    if not finite.any():
        raise ValueError("No finite Lomb–Scargle powers.")

    available = np.where(finite, powers, -np.inf).copy()
    peaks = []
    for _ in range(N_PEAKS):
        peak = int(np.argmax(available))
        if not np.isfinite(available[peak]):
            break
        peaks.append((float(1 / frequency[peak]), float(powers[peak])))
        available[np.abs(frequency - frequency[peak]) <= 1 / baseline] = -np.inf

    best_period, best_power = peaks[0]
    result = {"periodogram": periodogram, "peaks": peaks, "period": best_period, "power": best_power,
              "search_max_period": max_period}

    periods = 1 / frequency
    for name, select in {
        "short": periods < 1, "medium": (periods >= 1) & (periods <= 30),
        "long": periods > 30,
    }.items():
        indices = np.flatnonzero(select & finite)
        chosen = indices[np.argmax(powers[indices])] if indices.size else None
        result[f"best_{name}_period"] = float(periods[chosen]) if chosen is not None else np.nan
        result[f"best_{name}_power"] = float(powers[chosen]) if chosen is not None else np.nan

    result["peak_ratio_2_1"] = peaks[1][1] / best_power if len(peaks) > 1 and best_power > 0 else np.nan
    median_power = np.median(powers[finite])
    result["peak_to_median"] = best_power / median_power if median_power > 0 else np.nan
    result["near_search_boundary"] = bool(
        1 / best_period - 1 / max_period <= 1 / baseline
        or 1 / MIN_PERIOD - 1 / best_period <= 1 / baseline
    )
    return result


def fourier_fit(time, mag, period, harmonics=(1, 2, 3, 4)):
    if not np.isfinite(period) or period <= 0:
        return None
    phase = 2 * np.pi * ((time - time.min()) / period % 1)
    columns = [np.ones(len(time))]
    for k in harmonics:
        columns.extend([np.cos(k * phase), np.sin(k * phase)])
    design = np.column_stack(columns)
    if len(time) <= design.shape[1] or np.linalg.matrix_rank(design) != design.shape[1]:
        return None
    coeff = np.linalg.lstsq(design, mag, rcond=None)[0]
    rss = np.sum((mag - design @ coeff) ** 2)
    bic = len(time) * np.log(max(rss / len(time), np.finfo(float).tiny)) + len(coeff) * np.log(len(time))
    amps = {k: np.hypot(*coeff[1 + 2 * i:3 + 2 * i]) for i, k in enumerate(harmonics)}
    phases = {k: np.arctan2(-coeff[2 + 2 * i], coeff[1 + 2 * i]) for i, k in enumerate(harmonics)}
    result = {"coeff": coeff, "bic": bic, "amps": amps, "phases": phases}
    if {1, 2, 3}.issubset(amps) and amps[1] > 0:
        result["R21"] = amps[2] / amps[1]
        result["R31"] = amps[3] / amps[1]
        result["phi21"] = (phases[2] - 2 * phases[1] + np.pi) % (2 * np.pi) - np.pi
        result["phi31"] = (phases[3] - 3 * phases[1] + np.pi) % (2 * np.pi) - np.pi
    return result


def resolve_period(time, mag, period, max_period):
    naive = fourier_fit(time, mag, period)
    result = {"period_resolved": period, "delta_bic": np.nan,
              "odd_harmonic_fraction": np.nan, "odd_even_depth_difference": np.nan}
    if naive is None:
        return result, naive, naive

    even = fourier_fit(time, mag, 2 * period, harmonics=(2, 4))
    full = fourier_fit(time, mag, 2 * period)
    if even is None or full is None:
        return result, naive, naive

    result["delta_bic"] = even["bic"] - full["bic"]
    amps = full["amps"]
    norm = np.sqrt(sum(amp ** 2 for amp in amps.values()))
    if norm > 0:
        result["odd_harmonic_fraction"] = np.hypot(amps[1], amps[3]) / norm

    grid = np.arange(1024) / 1024
    model = np.full(len(grid), full["coeff"][0])
    for k in range(1, N_HARMONICS + 1):
        model += full["coeff"][2 * k - 1] * np.cos(2 * np.pi * k * grid)
        model += full["coeff"][2 * k] * np.sin(2 * np.pi * k * grid)
    peak = int(np.argmax(model))
    result["odd_even_depth_difference"] = abs(model[peak] - model[(peak + 512) % 1024])

    if result["delta_bic"] > 2:
        result["period_resolved"] = 2 * period
        return result, naive, full
    return result, naive, naive

## Period-Free Features

In [10]:
def timescale_features(time, mag, errors):
    gaps = np.diff(time)
    delta = np.diff(mag)
    visit = (gaps > 0) & (gaps <= 1)
    pair_error = errors[:-1] ** 2 + errors[1:] ** 2
    adjusted = np.maximum(delta ** 2 - pair_error, 0)
    slopes = np.sqrt(adjusted[visit & np.isfinite(adjusted)]) / gaps[visit & np.isfinite(adjusted)]

    left, right = np.triu_indices(len(time), k=1)
    lags = time[right] - time[left]
    squared = (mag[right] - mag[left]) ** 2
    noise = errors[left] ** 2 + errors[right] ** 2
    usable = np.isfinite(noise)

    short = usable & (lags >= 0.05) & (lags <= 0.3)
    long = usable & (lags > 6)
    sf_short = float(np.mean((squared - noise)[short])) if short.any() else np.nan
    sf_long = float(np.mean((squared - noise)[long])) if long.any() else np.nan

    return {
        "sf_short": sf_short,
        "sf_long": sf_long,
        "sf_ratio": sf_short / sf_long if np.isfinite(sf_short) and np.isfinite(sf_long) and sf_long > 0 else np.nan,
        "max_visit_slope": float(np.max(slopes)) if len(slopes) else np.nan,
    }


def stetson_j(time, mag, errors):
    valid = np.isfinite(errors) & (errors > 0)
    if valid.sum() < 2:
        return np.nan
    normalized = np.full(len(mag), np.nan)
    n = valid.sum()
    mean = np.average(mag[valid], weights=1 / errors[valid] ** 2)
    normalized[valid] = np.sqrt(n / (n - 1)) * (mag[valid] - mean) / errors[valid]
    pairs = (np.diff(time) > 0) & (np.diff(time) <= 0.3)
    product = normalized[:-1] * normalized[1:]
    selected = product[pairs & np.isfinite(product)]
    return float(np.mean(np.sign(selected) * np.sqrt(np.abs(selected)))) if len(selected) else np.nan

## Extract Features

In [13]:
records = []
search_failures = []
light_curves = {}

for source_id, group in observations.groupby("source_id", sort=False):
    time = group["g_transit_time"].to_numpy(dtype=float)
    mag = group["g_transit_mag"].to_numpy(dtype=float)
    errors = group["magnitude_error"].to_numpy(dtype=float)
    light_curves[source_id] = (time, mag, errors)

    gaps = np.diff(time)
    visit_gaps = gaps[gaps > 1]
    n_visits = int(1 + len(visit_gaps))
    amplitude = group["g_transit_mag"].quantile(0.95) - group["g_transit_mag"].quantile(0.05)
    median_error = group["magnitude_error"].median()
    record = {
        "source_id": source_id,
        "magnitude_mean": mag.mean(),
        "magnitude_std": mag.std(ddof=1) if len(mag) > 1 else np.nan,
        "magnitude_skew": group["g_transit_mag"].skew(),
        "magnitude_kurt": group["g_transit_mag"].kurt(),
        "magnitude_mad": np.median(np.abs(mag - np.median(mag))),
        "amplitude": amplitude,
        "median_error": median_error,
        "amplitude_ratio": amplitude / median_error if np.isfinite(median_error) and median_error > 0 else np.nan,
        "n_epochs": len(time),
        "time_span": np.ptp(time),
        "median_gap": np.median(gaps) if len(gaps) else np.nan,
        "short_gap_fraction": np.mean(gaps < 1) if len(gaps) else np.nan,
        "n_visits": n_visits,
        "epochs_per_visit": len(time) / n_visits,
        "median_visit_gap": np.median(visit_gaps) if len(visit_gaps) else np.nan,
        "visit_gap_cv": np.std(visit_gaps, ddof=1) / np.mean(visit_gaps) if len(visit_gaps) > 1 else np.nan,
        "searchable": False,
        "period_estimate": np.nan,
        "period_resolved": np.nan,
        "log_period": np.nan,
        "power": np.nan,
        "half_power": np.nan,
        "double_power": np.nan,
        "half_power_ratio": np.nan,
        "double_power_ratio": np.nan,
        "fourier_amp": np.nan,
        "fourier_ratio": np.nan,
        "fourier_amp_resolved": np.nan,
        "fourier_ratio_resolved": np.nan,
        "delta_bic": np.nan,
        "odd_harmonic_fraction": np.nan,
        "odd_even_depth_difference": np.nan,
        "search_max_period": np.nan,
        "near_search_boundary": np.nan,
        "peak_ratio_2_1": np.nan,
        "peak_to_median": np.nan,
        "beyond_1sigma_fraction": np.mean(np.abs(mag - np.median(mag)) > mag.std()) if mag.std() > 0 else np.nan,
        "von_neumann_eta": np.mean(np.diff(mag) ** 2) / np.var(mag, ddof=1) if len(mag) > 1 and np.var(mag, ddof=1) > 0 else np.nan,
        "stetson_j": stetson_j(time, mag, errors),
    }
    record.update(timescale_features(time, mag, errors))

    for k in range(1, N_PEAKS + 1):
        record[f"peak_period_{k}"] = np.nan
        record[f"peak_power_{k}"] = np.nan
    for band in ["short", "medium", "long"]:
        record[f"best_{band}_period"] = np.nan
        record[f"best_{band}_power"] = np.nan
    for period_type in ["naive", "resolved"]:
        for field in ["R21", "phi21", "R31", "phi31"]:
            record[f"{field}_{period_type}"] = np.nan

    try:
        search = period_search(time, mag, errors)
        if search is not None:
            period = search["period"]
            power = search["power"]
            f = 1 / period
            record["searchable"] = True
            record["period_estimate"] = period
            record["log_period"] = np.log10(period)
            record["power"] = power
            # Deleted: grid-bound checks around half- and double-period power.
            record["half_power"] = float(search["periodogram"].power(2 * f))
            record["double_power"] = float(search["periodogram"].power(f / 2))
            record["half_power_ratio"] = record["half_power"] / power if power > 0 else np.nan
            record["double_power_ratio"] = record["double_power"] / power if power > 0 else np.nan
            record["search_max_period"] = search["search_max_period"]
            record["near_search_boundary"] = search["near_search_boundary"]
            for name in ["peak_ratio_2_1", "peak_to_median"]:
                record[name] = search[name]
            for name in ["short", "medium", "long"]:
                record[f"best_{name}_period"] = search[f"best_{name}_period"]
                record[f"best_{name}_power"] = search[f"best_{name}_power"]
            for k, (peak_period, peak_power) in enumerate(search["peaks"], 1):
                record[f"peak_period_{k}"] = peak_period
                record[f"peak_power_{k}"] = peak_power

            resolution, naive, resolved = resolve_period(time, mag, period, search["search_max_period"])
            record.update(resolution)
            if naive is not None:
                record["fourier_amp"] = naive["amps"].get(1, np.nan)
                record["fourier_ratio"] = naive.get("R21", np.nan)
            if resolved is not None:
                record["fourier_amp_resolved"] = resolved["amps"].get(1, np.nan)
                record["fourier_ratio_resolved"] = resolved.get("R21", np.nan)
            for label, fit in [("naive", naive), ("resolved", resolved)]:
                if fit is not None:
                    for field in ["R21", "phi21", "R31", "phi31"]:
                        record[f"{field}_{label}"] = fit.get(field, np.nan)

    except (ValueError, FloatingPointError, ZeroDivisionError, np.linalg.LinAlgError) as exc:
        search_failures.append((source_id, f"{type(exc).__name__}: {exc}"))
    records.append(record)

## Check Extraction

In [14]:
if search_failures:
    raise RuntimeError(f"Period searches failed for {len(search_failures)} sources: {search_failures[:5]}")

if len(records) != len(sources) or sum(r["n_epochs"] for r in records) != len(observations):
    raise RuntimeError("Extracted source or epoch counts do not match CLEAN_FILE.")

if any(r["searchable"] and (not np.isfinite(r["period_estimate"]) or not np.isfinite(r["power"])) for r in records):
    raise RuntimeError("Searchable sources have invalid period-search results.")

if not all(np.isfinite(r["amplitude"]) and r["amplitude"] >= 0 for r in records):
    raise RuntimeError("Invalid photometric amplitudes.")

bad_periods = [
    (r["source_id"], r["period_estimate"], r["search_max_period"], r["log_period"])
    for r in records if r["searchable"] and not (
        (r["period_estimate"] >= MIN_PERIOD or np.isclose(r["period_estimate"], MIN_PERIOD, rtol=1e-10, atol=0))
        and (r["period_estimate"] <= r["search_max_period"] or np.isclose(r["period_estimate"], r["search_max_period"], rtol=1e-10, atol=0))
        and np.isclose(np.log10(r["period_estimate"]), r["log_period"], rtol=0, atol=1e-10)
    )
]

if bad_periods:
    raise RuntimeError(f"Invalid period boundaries or log_period for {len(bad_periods)} sources: {bad_periods[:5]}")

print("Extracted Sources:", len(records))
print("Searchable Sources:", sum(r["searchable"] for r in records))

Extracted Sources: 2000
Searchable Sources: 1978


## Feature Tables

In [15]:
stars = sources[["source_id", "class_name", "bp_rp_color", "ecl_lat", "ra", "dec",
                 "g_mean_magnitude", "gaia_period", "gaia_period_origin"]].merge(
    pd.DataFrame(records), on="source_id", how="left", validate="one_to_one"
)
stars["abs_latitude"] = stars["ecl_lat"].abs()
stars["bp_rp_missing"] = stars["bp_rp_color"].isna().astype(int)

primary_cols = [
    "magnitude_mean", "magnitude_std", "magnitude_skew", "magnitude_kurt",
    "amplitude", "log_period", "power", "fourier_amp", "fourier_ratio",
    "bp_rp_color", "magnitude_mad", "median_error", "amplitude_ratio", "half_power", "double_power",
]
additional_cols = [
    "half_power_ratio", "double_power_ratio", "peak_ratio_2_1", "peak_to_median",
    "period_resolved", "delta_bic", "odd_harmonic_fraction", "odd_even_depth_difference",
    "fourier_amp_resolved", "fourier_ratio_resolved",
    "beyond_1sigma_fraction", "von_neumann_eta", "stetson_j", "sf_short", "sf_long",
    "sf_ratio", "max_visit_slope",
    *[f"peak_{kind}_{k}" for k in range(1, N_PEAKS + 1) for kind in ["period", "power"]],
    *[f"best_{band}_{kind}" for band in ["short", "medium", "long"] for kind in ["period", "power"]],
    *[f"{field}_{label}" for label in ["naive", "resolved"]
      for field in ["R21", "phi21", "R31", "phi31"]],
]
sampling_cols = [
    "n_epochs", "time_span", "median_gap", "short_gap_fraction",
    "n_visits", "epochs_per_visit", "median_visit_gap", "visit_gap_cv",
]
predictor_cols = primary_cols + additional_cols
stars[predictor_cols + sampling_cols] = stars[predictor_cols + sampling_cols].replace(
    [np.inf, -np.inf], np.nan
)

model_sources = stars["searchable"].fillna(False)
features = stars.loc[model_sources, ["source_id", "class_name", *predictor_cols]].copy()
ablation = stars.loc[model_sources, ["source_id", "class_name", *predictor_cols,
                                    *sampling_cols, "bp_rp_missing"]].copy()

excluded = {"gaia_period", "gaia_period_origin", "abs_latitude", "period_eligible",
            "period_status", "half_period_alias", "double_period_alias", "recovered"}
if excluded.intersection(features.columns):
    raise ValueError("Validation-only columns entered the classification predictors.")
if not all(pd.api.types.is_numeric_dtype(features[col]) for col in predictor_cols):
    raise ValueError("Nonnumeric predictors.")
if np.isinf(features[predictor_cols].to_numpy(dtype=float)).any():
    raise ValueError("Infinite predictors.")
if not features["source_id"].is_unique:
    raise ValueError("Duplicate model rows.")

ineligible = stars.loc[~model_sources, ["source_id", "class_name", "n_epochs"]]
print("Excluded from Classification:", len(ineligible))
display(ineligible.groupby("class_name").size().reindex(CLASSES, fill_value=0).rename("excluded"))

Excluded from Classification: 22


class_name
RR Lyrae                22
Cepheid                  0
Eclipsing Binary         0
Long-Period Variable     0
Name: excluded, dtype: int64

## Period Recovery

In [16]:
def period_match(estimate, reference, span):
    estimate, reference, span = np.broadcast_arrays(
        np.asarray(estimate, dtype=float),
        np.asarray(reference, dtype=float),
        np.asarray(span, dtype=float)
    )
    valid = (
        np.isfinite(estimate) & (estimate > 0)
        & np.isfinite(reference) & (reference > 0)
        & np.isfinite(span) & (span > 0)
    )
    with np.errstate(divide="ignore", invalid="ignore"):
        drift = span * np.abs(1 / estimate - 1 / reference)
    return valid & (drift <= MAX_DRIFT)

estimate = stars["period_estimate"].to_numpy(dtype=float)
published = stars["gaia_period"].to_numpy(dtype=float)
span = stars["time_span"].to_numpy(dtype=float)
maximum = stars["search_max_period"].to_numpy(dtype=float)

eligible = (
    stars["searchable"].fillna(False).to_numpy(dtype=bool)
    & np.isfinite(published) & (published >= MIN_PERIOD)
    & (published <= maximum)
)
exact = eligible & period_match(estimate, published, span)
half = eligible & ~exact & period_match(estimate, published / 2, span)
double = eligible & ~exact & ~half & period_match(estimate, published * 2, span)

stars["period_eligible"] = eligible
stars["phase_coherent_recovery"] = exact
stars["half_period_alias"] = half
stars["double_period_alias"] = double
stars["period_status"] = np.select(
    [~eligible, exact, half, double],
    ["ineligible", "exact", "half", "double"], default="mismatch"
)
with np.errstate(divide="ignore", invalid="ignore"):
    stars["primary_relative_error"] = np.where(
        eligible, np.abs(estimate / published - 1), np.nan
    )
    stars["primary_phase_drift_cycles"] = np.where(
        eligible, span * np.abs(1 / estimate - 1 / published), np.nan
    )
stars["relative_period_accurate"] = (
    eligible & (stars["primary_relative_error"].to_numpy() <= 0.10)
)

for k in [1, 3, 5]:
    recovered = np.zeros(len(stars), dtype=bool)
    for j in range(1, k + 1):
        peak = stars[f"peak_period_{j}"].to_numpy(dtype=float)
        recovered |= period_match(peak, published, span)
    stars[f"recovery_at_{k}"] = eligible & recovered

recovery = pd.crosstab(
    stars.loc[eligible, "class_name"], stars.loc[eligible, "period_status"]
).reindex(index=CLASSES, columns=["exact", "half", "double", "mismatch"], fill_value=0)

display(recovery)
display(
    stars.loc[eligible].groupby("class_name")[
        ["recovery_at_1", "recovery_at_3", "recovery_at_5"]
    ].mean().reindex(CLASSES).round(3)
)

period_status,exact,half,double,mismatch
class_name,,,,
RR Lyrae,392,1,0,85
Cepheid,495,0,0,5
Eclipsing Binary,12,366,2,120
Long-Period Variable,269,0,0,132


,recovery_at_1,recovery_at_3,recovery_at_5
class_name,,,
RR Lyrae,0.820,0.921,0.948
Cepheid,0.990,0.996,0.996
Eclipsing Binary,0.024,0.054,0.062
Long-Period Variable,0.671,0.805,0.863


## Sky Position and Observation Quality

In [17]:
healpix = HEALPix(nside=16, order="ring", frame="icrs")
sky_cell = np.full(len(stars), -1, dtype=int)
valid_sky = stars["ra"].between(0, 360) & stars["dec"].between(-90, 90)
sky_cell[valid_sky] = healpix.lonlat_to_healpix(
    stars.loc[valid_sky, "ra"].to_numpy() * u.deg,
    stars.loc[valid_sky, "dec"].to_numpy() * u.deg
)
stars["sky_cell"] = sky_cell

raw = pd.read_csv(
    PHOTOMETRY_FILE,
    usecols=lambda col: col in {
        "source_id", "band", "variability_flag_g_reject", "rejected_by_variability"
    }
)
if "band" in raw:
    raw = raw.loc[raw["band"].astype("string").str.upper().eq("G")]

flag_col = (
    "variability_flag_g_reject" if "variability_flag_g_reject" in raw
    else "rejected_by_variability"
)
flags = raw[flag_col].astype("string").str.strip().str.lower()
raw["rejected"] = np.select(
    [flags.isin(["true", "1"]), flags.isin(["false", "0"])],
    [1.0, 0.0], default=np.nan
)
rejection = raw.groupby("source_id")["rejected"].mean()
stars["gaia_rejection_fraction"] = stars["source_id"].map(rejection)

print("Sky cells:", stars.loc[stars["sky_cell"].ge(0), "sky_cell"].nunique())
print("Sources with rejection data:", stars["gaia_rejection_fraction"].notna().sum())

Sky cells: 169
Sources with rejection data: 2000


## Period Validation

In [18]:
with np.errstate(divide="ignore", invalid="ignore"):
    stars["n_cycles"] = stars["time_span"] / stars["gaia_period"]
    stars["epochs_per_cycle"] = stars["n_epochs"] / stars["n_cycles"]
    stars["visits_per_cycle"] = stars["n_visits"] / stars["n_cycles"]

validation_cols = [
    "source_id", "class_name", "gaia_period", "gaia_period_origin",
    "period_estimate", "period_resolved", "g_mean_magnitude",
    "ecl_lat", "abs_latitude", "sky_cell", "gaia_rejection_fraction",
    "n_cycles", "epochs_per_cycle", "visits_per_cycle",
    *sampling_cols, "search_max_period", "near_search_boundary",
    "period_eligible", "primary_relative_error",
    "primary_phase_drift_cycles", "relative_period_accurate",
    "phase_coherent_recovery", "half_period_alias", "double_period_alias",
    "period_status", "recovery_at_1", "recovery_at_3", "recovery_at_5",
    *[f"peak_{kind}_{k}" for k in range(1, N_PEAKS + 1) for kind in ["period", "power"]]
]
period_validation = (
    stars[validation_cols].replace([np.inf, -np.inf], np.nan).copy()
)

display(
    period_validation.loc[period_validation["period_eligible"]]
    .groupby("class_name")
    .agg(
        stars=("source_id", "size"),
        median_cycles=("n_cycles", "median"),
        median_epochs=("n_epochs", "median"),
        recovery=("phase_coherent_recovery", "mean")
    ).reindex(CLASSES).round(3)
)

,stars,median_cycles,median_epochs,recovery
class_name,,,,
RR Lyrae,478,1967.233,37.0,0.820
Cepheid,500,305.746,39.0,0.990
Eclipsing Binary,500,2243.189,40.0,0.024
Long-Period Variable,401,5.183,42.0,0.671


## Oracle Features

In [19]:
oracle_records = []
for _, star in stars.loc[eligible].iterrows():
    time, mag, errors = light_curves[star["source_id"]]
    period = float(star["gaia_period"])
    maximum = float(star["search_max_period"])
    search = LombScargle(time - time.min(), mag, normalization="standard")
    power = float(search.power(1 / period))
    half_power = float(search.power(2 / period))
    double_power = float(search.power(1 / (2 * period)))
    resolution, naive, resolved = resolve_period(time, mag, period, maximum)
    row = {
        "source_id": star["source_id"], "class_name": star["class_name"],
        "gaia_period": period, "log_period": np.log10(period), "power": power,
        "half_power": half_power, "double_power": double_power,
        "half_power_ratio": half_power / power if power > 0 else np.nan,
        "double_power_ratio": double_power / power if power > 0 else np.nan,
        "period_resolved": resolution["period_resolved"],
        "delta_bic": resolution["delta_bic"],
        "odd_harmonic_fraction": resolution["odd_harmonic_fraction"],
        "odd_even_depth_difference": resolution["odd_even_depth_difference"],
        "fourier_amp": naive["amps"].get(1, np.nan) if naive else np.nan,
        "fourier_ratio": naive.get("R21", np.nan) if naive else np.nan,
        "fourier_amp_resolved": resolved["amps"].get(1, np.nan) if resolved else np.nan,
        "fourier_ratio_resolved": resolved.get("R21", np.nan) if resolved else np.nan,
    }
    for label, fit in [("naive", naive), ("resolved", resolved)]:
        for field in ["R21", "phi21", "R31", "phi31"]:
            row[f"{field}_{label}"] = fit.get(field, np.nan) if fit else np.nan
    oracle_records.append(row)

oracle = pd.DataFrame(oracle_records).replace([np.inf, -np.inf], np.nan)


print("Oracle sources:", len(oracle))
display(oracle.groupby("class_name").size().reindex(CLASSES, fill_value=0))

Oracle sources: 1879


class_name
RR Lyrae                478
Cepheid                 500
Eclipsing Binary        500
Long-Period Variable    401
dtype: int64

## Check Features

In [20]:
diagnostics = pd.DataFrame({
    "feature": predictor_cols,
    "missing": features[predictor_cols].isna().sum().to_numpy(),
    "missing_pct": 100 * features[predictor_cols].isna().mean().to_numpy()
})
display(diagnostics.sort_values("missing_pct", ascending=False).round(2))

corr = features[predictor_cols].corr(method="spearman").abs()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
display(pairs[pairs.ge(CORR_LIMIT)].sort_values(ascending=False).rename("abs_corr").to_frame())

if features["source_id"].duplicated().any():
    raise ValueError("Duplicate classification sources.")
if not features["source_id"].isin(stars.loc[stars["searchable"], "source_id"]).all():
    raise ValueError("Ineligible sources entered classification.")
if np.isinf(features[predictor_cols].to_numpy(dtype=float)).any():
    raise ValueError("Infinite classification predictors.")
if set(oracle["source_id"]) != set(stars.loc[stars["period_eligible"], "source_id"]):
    raise ValueError("Oracle coverage differs from eligible sources.")

print("Complete predictor rows:", features[predictor_cols].notna().all(axis=1).sum())

,feature,missing,missing_pct
9,bp_rp_color,39,1.97
30,sf_ratio,30,1.52
42,best_short_period,0,0.00
31,max_visit_slope,0,0.00
32,peak_period_1,0,0.00
33,peak_power_1,0,0.00
34,peak_period_2,0,0.00
35,peak_power_2,0,0.00
36,peak_period_3,0,0.00
37,peak_power_3,0,0.00


abs_corr
fourier_ratio          R21_naive           1.000000
fourier_ratio_resolved R21_resolved        1.000000
power                  peak_power_1        1.000000
log_period             peak_period_1       1.000000
                       period_resolved     0.988947
period_resolved        peak_period_1       0.988947
peak_power_4           peak_power_5        0.982608
magnitude_std          amplitude           0.979082
double_power           double_power_ratio  0.968426
peak_power_3           peak_power_4        0.966607
magnitude_std          sf_long             0.962214
half_power             half_power_ratio    0.952338
peak_power_3           peak_power_5        0.949292
peak_power_2           peak_power_3        0.942083
amplitude              sf_long             0.940658
magnitude_std          fourier_amp         0.933166
sf_short               max_visit_slope     0.924029
fourier_amp            sf_long             0.910773
magnitude_mean         median_error        0.910096
peak_power_2           peak_power_4        0.908188
amplitude              fourier_amp         0.904037

Complete predictor rows: 1915


## Correlation Screening

In [21]:
def select_features(data, columns, limit=CORR_LIMIT):
    correlation = data[columns].corr(method="spearman").abs()
    remaining = set(columns)
    selected = []
    while remaining:
        first = next(col for col in columns if col in remaining)
        cluster = {first}
        pending = [first]
        while pending:
            current = pending.pop()
            neighbors = (
                set(correlation.index[correlation.loc[current].ge(limit)]) & remaining
            )
            new = neighbors - cluster
            cluster.update(new)
            pending.extend(new)
        best = max(cluster, key=lambda col: (
            data[col].notna().sum(), -columns.index(col)
        ))
        selected.append(best)
        remaining -= cluster
    return selected

linear_svm_cols = select_features(features, predictor_cols)
selected_features = pd.DataFrame({
    "feature": predictor_cols,
    "selected": [col in linear_svm_cols for col in predictor_cols]
})
print("Original predictors:", len(predictor_cols))
print("Selected predictors:", len(linear_svm_cols))
display(selected_features.loc[selected_features["selected"]])

Original predictors: 56
Selected predictors: 41


,feature,selected
0,magnitude_mean,True
1,magnitude_std,True
2,magnitude_skew,True
3,magnitude_kurt,True
5,log_period,True
6,power,True
8,fourier_ratio,True
9,bp_rp_color,True
10,magnitude_mad,True
12,amplitude_ratio,True


## Wrong-Period Classification

In [22]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

period_cols = [
    "log_period", "power", "half_power", "double_power",
    "half_power_ratio", "double_power_ratio",
    "fourier_amp", "fourier_ratio", "period_resolved", "delta_bic",
    "odd_harmonic_fraction", "odd_even_depth_difference",
    "fourier_amp_resolved", "fourier_ratio_resolved",
    *[f"{field}_{label}" for label in ["naive", "resolved"]
      for field in ["R21", "phi21", "R31", "phi31"]]
]
free_cols = [
    col for col in predictor_cols
    if col not in period_cols and not col.startswith(("peak_", "best_"))
]
color_cols = ["bp_rp_color", "amplitude"]
shape_cols = [
    "bp_rp_color", "amplitude", "magnitude_skew", "magnitude_kurt",
    "magnitude_mad", "beyond_1sigma_fraction", "von_neumann_eta",
    "stetson_j", "sf_short", "sf_long", "sf_ratio", "max_visit_slope"
]
wrong = stars.loc[
    stars["period_eligible"] & stars["period_status"].ne("exact")
].copy()
wrong_features = features.loc[features["source_id"].isin(wrong["source_id"])].copy()

print("Wrong-period sources:", len(wrong_features))
display(wrong["period_status"].value_counts())
display(wrong["class_name"].value_counts().reindex(CLASSES, fill_value=0))

Wrong-period sources: 711


period_status
half        367
mismatch    342
double        2
Name: count, dtype: int64

class_name
RR Lyrae                 86
Cepheid                   5
Eclipsing Binary        488
Long-Period Variable    132
Name: count, dtype: int64

## Classification Models

In [23]:
def make_model(name, seed=SEED):
    if name == "Logistic":
        estimator = LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=seed
        )
    elif name == "SVM":
        estimator = SVC(kernel="rbf", C=1, class_weight="balanced", random_state=seed)
    elif name == "Forest":
        estimator = RandomForestClassifier(
            n_estimators=300, min_samples_leaf=2,
            class_weight="balanced", random_state=seed, n_jobs=1
        )
    else:
        raise ValueError(f"Unknown model: {name}")

    steps = [("imputer", SimpleImputer(strategy="median", keep_empty_features=True))]
    if name != "Forest":
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)


def score_model(train, test, columns, name):
    chosen = select_features(train, columns) if name in ["Logistic", "SVM"] else columns
    model = make_model(name)
    model.fit(train[chosen], train["class_name"])
    predicted = model.predict(test[chosen])
    return {
        "accuracy": accuracy_score(test["class_name"], predicted),
        "balanced_accuracy": balanced_accuracy_score(test["class_name"], predicted),
        "macro_f1": f1_score(
            test["class_name"], predicted,
            labels=CLASSES, average="macro", zero_division=0
        ),
        "n_features": len(chosen)
    }, predicted

## Classification Experiment

In [24]:
comparison = features.merge(
    period_validation[["source_id", "period_eligible", "period_status"]],
    on="source_id", validate="one_to_one"
)
comparison = comparison.loc[comparison["period_eligible"]].copy()

# Peak-ranking features have no Gaia-period counterpart; exclude them from paired oracle tests.
oracle_cols = [col for col in period_cols if col in oracle.columns and col in features.columns]
compare_cols = [col for col in predictor_cols if col in free_cols or col in oracle_cols]
oracle_data = comparison.merge(
    oracle[["source_id", *oracle_cols]],
    on="source_id", how="left", validate="one_to_one", suffixes=("", "_oracle")
)
for col in oracle_cols:
    oracle_data[col] = oracle_data.pop(f"{col}_oracle")

splits = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=SEED)
experiments = {
    "color_amplitude": color_cols,
    "period_free": free_cols,
    "shape_free": shape_cols,
    "full": predictor_cols
}

results = []
predictions = []
for split, (train_idx, test_idx) in enumerate(
    splits.split(comparison, comparison["class_name"]), 1
):
    train, test = comparison.iloc[train_idx], comparison.iloc[test_idx]
    oracle_train, oracle_test = oracle_data.iloc[train_idx], oracle_data.iloc[test_idx]

    for name in ["Logistic", "SVM", "Forest"]:
        for experiment, columns in [*experiments.items(), ("oracle_comparison", compare_cols)]:
            for source in (["estimated", "oracle"] if experiment == "oracle_comparison" else ["estimated"]):
                a, b = (oracle_train, oracle_test) if source == "oracle" else (train, test)
                scores, predicted = score_model(a, b, columns, name)

                for group, mask in {
                    "all": np.ones(len(b), dtype=bool),
                    "correct": b["period_status"].eq("exact").to_numpy(),
                    "wrong": b["period_status"].ne("exact").to_numpy()
                }.items():
                    if not mask.any():
                        continue
                    truth = b["class_name"].to_numpy()[mask]
                    labels = predicted[mask]
                    results.append({
                        "split": split, "model": name, "experiment": experiment,
                        "period_source": source, "group": group, "n_stars": len(truth),
                        "accuracy": accuracy_score(truth, labels),
                        "balanced_accuracy": balanced_accuracy_score(truth, labels),
                        "macro_f1": f1_score(
                            truth, labels, labels=CLASSES, average="macro", zero_division=0
                        ),
                        "n_features": scores["n_features"]
                    })

                predictions.extend({
                    "split": split, "model": name, "experiment": experiment,
                    "period_source": source, "source_id": source_id,
                    "class_name": truth, "predicted": label, "period_status": status
                } for source_id, truth, label, status in zip(
                    b["source_id"], b["class_name"], predicted, b["period_status"]
                ))

classification = pd.DataFrame(results)
predictions = pd.DataFrame(predictions)
display(
    classification.groupby(["model", "experiment", "period_source", "group"])[
        ["accuracy", "balanced_accuracy", "macro_f1"]
    ].mean().round(3)
)

/Users/Sumaitat/Documents/Coding/gaia-varstars/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/Sumaitat/Documents/Coding/gaia-varstars/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/Sumaitat/Documents/Coding/gaia-varstars/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/Sumaitat/Documents/Coding/gaia-varstars/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2939: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/Users/Sumaitat/Documents/Coding/gaia-varstars/.venv/lib/python3.12/site-pac

accuracy  balanced_accuracy  \
model    experiment        period_source group                                  
Forest   color_amplitude   estimated     all         0.597              0.610   
                                         correct     0.551              0.618   
                                         wrong       0.672              0.579   
         full              estimated     all         0.974              0.975   
                                         correct     0.971              0.981   
                                         wrong       0.979              0.959   
         oracle_comparison estimated     all         0.971              0.972   
                                         correct     0.966              0.977   
                                         wrong       0.978              0.907   
                           oracle        all         0.982              0.983   
                                         correct     0.975              0.899   
                                         wrong       0.993              0.921   
         period_free       estimated     all         0.951              0.952   
                                         correct     0.940              0.955   
                                         wrong       0.968              0.905   
         shape_free        estimated     all         0.937              0.939   
                                         correct     0.922              0.930   
                                         wrong       0.961              0.896   
Logistic color_amplitude   estimated     all         0.489              0.508   
                                         correct     0.404              0.532   
                                         wrong       0.624              0.465   
         full              estimated     all         0.950              0.951   
                                         correct     0.950              0.938   
                                         wrong       0.951              0.959   
         oracle_comparison estimated     all         0.940              0.942   
                                         correct     0.940              0.945   
                                         wrong       0.940              0.877   
                           oracle        all         0.976              0.977   
                                         correct     0.967              0.904   
                                         wrong       0.990              0.920   
         period_free       estimated     all         0.895              0.895   
                                         correct     0.871              0.900   
                                         wrong       0.934              0.872   
         shape_free        estimated     all         0.887              0.888   
                                         correct     0.858              0.881   
                                         wrong       0.933              0.867   
SVM      color_amplitude   estimated     all         0.579              0.596   
                                         correct     0.560              0.619   
                                         wrong       0.609              0.538   
         full              estimated     all         0.958              0.960   
                                         correct     0.954              0.958   
                                         wrong       0.964              0.972   
         oracle_comparison estimated     all         0.958              0.960   
                                         correct     0.956              0.958   
                                         wrong       0.962              0.975   
                           oracle        all         0.973              0.973   
                                         correct     0.964              0.878   
                                         wrong       0.988              0.915   


## Wrong-Period Results

In [25]:
wrong_predictions = predictions.loc[predictions["period_status"].ne("exact")].copy()
wrong_predictions["correct"] = (
    wrong_predictions["class_name"] == wrong_predictions["predicted"]
)
wrong_summary = (
    wrong_predictions.groupby(["model", "experiment", "period_source", "class_name"])
    .agg(predictions=("correct", "size"), accuracy=("correct", "mean"))
    .reset_index()
)
display(
    wrong_summary.pivot_table(
        index=["model", "experiment", "period_source"],
        columns="class_name", values="accuracy"
    ).round(3)
)

oracle_scores = classification.loc[
    classification["experiment"].eq("oracle_comparison")
].pivot_table(
    index=["split", "model", "group"],
    columns="period_source", values="balanced_accuracy"
)
oracle_scores["oracle_gain"] = (
    oracle_scores["oracle"] - oracle_scores["estimated"]
)
display(oracle_scores.groupby(["model", "group"]).mean().round(3))

class_name                                Cepheid  Eclipsing Binary  \
model    experiment        period_source                              
Forest   color_amplitude   estimated          0.2             0.667   
         full              estimated          0.8             0.982   
         oracle_comparison estimated          0.6             0.984   
                           oracle             0.6             0.996   
         period_free       estimated          0.6             0.967   
         shape_free        estimated          0.6             0.965   
Logistic color_amplitude   estimated          0.0             0.649   
         full              estimated          1.0             0.955   
         oracle_comparison estimated          0.6             0.945   
                           oracle             0.6             0.992   
         period_free       estimated          0.6             0.943   
         shape_free        estimated          0.6             0.945   
SVM      color_amplitude   estimated          0.0             0.540   
         full              estimated          1.0             0.963   
         oracle_comparison estimated          1.0             0.955   
                           oracle             0.6             0.994   
         period_free       estimated          0.6             0.959   
         shape_free        estimated          0.6             0.955   

class_name                                Long-Period Variable  RR Lyrae  
model    experiment        period_source                                  
Forest   color_amplitude   estimated                     0.910     0.408  
         full              estimated                     0.985     0.971  
         oracle_comparison estimated                     0.977     0.971  
                           oracle                        1.000     0.990  
         period_free       estimated                     0.985     0.971  
         shape_free        estimated                     0.962     0.961  
Logistic color_amplitude   estimated                     0.872     0.214  
         full              estimated                     0.955     0.922  
         oracle_comparison estimated                     0.970     0.893  
                           oracle                        1.000     0.990  
         period_free       estimated                     0.925     0.922  
         shape_free        estimated                     0.910     0.922  
SVM      color_amplitude   estimated                     0.962     0.505  
         full              estimated                     1.000     0.922  
         oracle_comparison estimated                     0.992     0.951  
                           oracle                        0.985     0.981  
         period_free       estimated                     1.000     0.932  
         shape_free        estimated                     0.992     0.903

period_source     estimated  oracle  oracle_gain
model    group                                  
Forest   all          0.972   0.983        0.011
         correct      0.977   0.899       -0.077
         wrong        0.907   0.921        0.014
Logistic all          0.942   0.977        0.035
         correct      0.945   0.904       -0.042
         wrong        0.877   0.920        0.043
SVM      all          0.960   0.973        0.014
         correct      0.958   0.878       -0.080
         wrong        0.975   0.915       -0.059

## Epoch Subsampling

In [26]:
def subsample_indices(time, count, rng, method="random"):
    n = len(time)
    if count >= n:
        return np.arange(n)
    if method == "random":
        return np.sort(rng.choice(n, count, replace=False))
    if method != "visits":
        raise ValueError(f"Unknown sampling method: {method}")

    # Prioritize complete observing visits before filling a partial final visit.
    visits = np.cumsum(np.r_[0, np.diff(time) > 1])
    blocks = [np.flatnonzero(visits == k) for k in np.unique(visits)]
    selected = []
    for j in rng.permutation(len(blocks)):
        block = blocks[j]
        if len(selected) + len(block) <= count:
            selected.extend(block.tolist())
        if len(selected) == count:
            break
    if len(selected) < count:
        remaining = np.setdiff1d(np.arange(n), np.asarray(selected, dtype=int))
        selected.extend(rng.choice(remaining, count - len(selected), replace=False).tolist())
    return np.sort(np.asarray(selected, dtype=int))


def subsample_period(source_id, count, repeat, method):
    time, mag, errors = light_curves[source_id]
    seed = np.random.SeedSequence([
        SEED, int(source_id) % (2 ** 32), int(source_id) // (2 ** 32),
        int(count), int(repeat) + 1, int(method == "visits")
    ])
    rng = np.random.default_rng(seed)
    indices = subsample_indices(time, count, rng, method)
    sub_time, sub_mag, sub_errors = time[indices], mag[indices], errors[indices]
    search = period_search(sub_time, sub_mag, sub_errors)
    period = search["period"] if search else np.nan
    reference = float(reference_periods[source_id])
    maximum = search["search_max_period"] if search else np.nan
    eligible_trial = search is not None and MIN_PERIOD <= reference <= maximum
    recovered = bool(eligible_trial and period_match(period, reference, np.ptp(sub_time)))
    return {
        "source_id": source_id, "epochs": len(indices),
        "baseline": np.ptp(sub_time), "period_estimate": period,
        "period_eligible": eligible_trial, "recovered": recovered,
        "method": method, "repeat": repeat
    }

## Set Up Subsampling

In [27]:
levels = [20, 25, 30, 40, 50]
repetitions = 10
control = stars.loc[
    stars["period_eligible"] & stars["n_epochs"].ge(50),
    ["source_id", "class_name", "gaia_period", "n_epochs"]
].copy()
reference_periods = control.set_index("source_id")["gaia_period"].to_dict()

display(
    control["class_name"].value_counts()
    .reindex(CLASSES, fill_value=0).rename("control_stars")
)

jobs = [
    (row.source_id, count, repeat, method)
    for row in control.itertuples()
    for count in levels
    for method in ["random", "visits"]
    for repeat in range(repetitions)
]
print("Subsampling trials:", len(jobs))

class_name
RR Lyrae                 98
Cepheid                 106
Eclipsing Binary        145
Long-Period Variable    115
Name: control_stars, dtype: int64

Subsampling trials: 46400


## Run Subsampling

In [28]:
trials = Parallel(n_jobs=4, prefer="threads")(
    delayed(subsample_period)(*job) for job in jobs
)
full_trials = Parallel(n_jobs=4, prefer="threads")(
    delayed(subsample_period)(row.source_id, int(row.n_epochs), 0, "full")
    for row in control.itertuples()
)
subsampling = pd.DataFrame(trials + full_trials).merge(
    control[["source_id", "class_name"]], on="source_id", validate="many_to_one"
)
print("Completed trials:", len(subsampling))

Completed trials: 46864


## Check Subsampling

In [29]:
full = subsampling.loc[subsampling["method"].eq("full")].copy()
natural = stars.set_index("source_id")["phase_coherent_recovery"]
full = full.set_index("source_id")
expected = natural.reindex(full.index)

if not full["recovered"].eq(expected).all():
    mismatches = full.index[~full["recovered"].eq(expected)]
    raise ValueError(f"Full-epoch recovery disagrees for {len(mismatches)} sources.")
if not full["period_eligible"].all():
    raise ValueError("Full-epoch control contains ineligible periods.")
if not subsampling["epochs"].ge(MIN_POINTS).all():
    raise ValueError("Subsampling produced curves below MIN_POINTS.")

sampled = subsampling.loc[subsampling["method"].ne("full")]
counts = sampled.groupby(["source_id", "epochs", "method"])["repeat"].nunique()
if not counts.eq(repetitions).all():
    raise ValueError("Missing or duplicate subsampling repetitions.")

print("Full-epoch consistency: passed")
print("Minimum-epoch check: passed")
print("Repetition check: passed")

Full-epoch consistency: passed
Minimum-epoch check: passed
Repetition check: passed


## Subsampling Results

In [30]:
star_rates = (
    subsampling.loc[subsampling["method"].ne("full")]
    .groupby(["class_name", "source_id", "epochs", "method"], as_index=False)
    ["recovered"].mean()
)


def bootstrap_stars(values, seed=SEED, draws=2000):
    values = np.asarray(values, dtype=float)
    if not len(values):
        return pd.Series({
            "n_stars": 0, "recovery_mean": np.nan,
            "ci_low": np.nan, "ci_high": np.nan
        })
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(values), size=(draws, len(values)))
    means = values[indices].mean(axis=1)
    return pd.Series({
        "n_stars": len(values), "recovery_mean": values.mean(),
        "ci_low": np.quantile(means, 0.025),
        "ci_high": np.quantile(means, 0.975)
    })

subsample_summary = (
    star_rates.groupby(["class_name", "epochs", "method"])["recovered"]
    .apply(bootstrap_stars).unstack().reset_index()
)
display(subsample_summary.round(3))

,class_name,epochs,method,n_stars,recovery_mean,ci_low,ci_high
0,Cepheid,20,random,106.0,0.882,0.834,0.924
1,Cepheid,20,visits,106.0,0.722,0.660,0.777
2,Cepheid,25,random,106.0,0.944,0.910,0.970
3,Cepheid,25,visits,106.0,0.840,0.789,0.886
4,Cepheid,30,random,106.0,0.958,0.927,0.983
5,Cepheid,30,visits,106.0,0.921,0.883,0.954
6,Cepheid,40,random,106.0,0.980,0.954,0.998
7,Cepheid,40,visits,106.0,0.962,0.934,0.984
8,Cepheid,50,random,106.0,0.987,0.963,1.000
9,Cepheid,50,visits,106.0,0.977,0.953,0.993


## Compare Natural Recovery

In [31]:
natural_all = (
    stars.loc[stars["period_eligible"]]
    .groupby("class_name")["phase_coherent_recovery"].mean()
)
natural_control = (
    control.assign(recovered=control["source_id"].map(natural))
    .groupby("class_name")["recovered"].mean()
)
recovery_40 = (
    star_rates.loc[star_rates["epochs"].eq(40) & star_rates["method"].eq("random")]
    .groupby("class_name")["recovered"].mean()
)
comparison_40 = pd.DataFrame({
    "natural_all": natural_all,
    "natural_control": natural_control,
    "subsample_40": recovery_40
})
comparison_40["cohort_difference"] = (
    comparison_40["natural_control"] - comparison_40["natural_all"]
)
comparison_40["sampling_difference"] = (
    comparison_40["subsample_40"] - comparison_40["natural_control"]
)
display(comparison_40.reindex(CLASSES).round(3))

,natural_all,natural_control,subsample_40,cohort_difference,sampling_difference
class_name,,,,,
RR Lyrae,0.820,0.939,0.868,0.119,-0.070
Cepheid,0.990,0.991,0.980,0.001,-0.010
Eclipsing Binary,0.024,0.021,0.005,-0.003,-0.016
Long-Period Variable,0.671,0.835,0.617,0.164,-0.217


## Sampling and Latitude

In [32]:
latitude_bins = [0, 15, 30, 45, 60, 75, 90]
epoch_bins = [20, 30, 40, 50, 75, 100, np.inf]
latitude_data = period_validation.loc[period_validation["period_eligible"]].copy()
latitude_data["latitude_bin"] = pd.cut(
    latitude_data["abs_latitude"], latitude_bins, include_lowest=True
)
latitude_data["epoch_bin"] = pd.cut(
    latitude_data["n_epochs"], epoch_bins, include_lowest=True
)
latitude_recovery = (
    latitude_data.groupby(["class_name", "latitude_bin"], observed=True)
    .agg(
        stars=("source_id", "size"),
        recovery=("phase_coherent_recovery", "mean"),
        median_epochs=("n_epochs", "median"),
        median_visits=("n_visits", "median")
    ).reset_index()
)
epoch_recovery = (
    latitude_data.groupby(["class_name", "epoch_bin"], observed=True)
    .agg(
        stars=("source_id", "size"),
        recovery=("phase_coherent_recovery", "mean"),
        median_latitude=("abs_latitude", "median")
    ).reset_index()
)
display(latitude_recovery.round(3))
display(epoch_recovery.round(3))

,class_name,latitude_bin,stars,recovery,median_epochs,median_visits
0,Cepheid,"(-0.001, 15.0]",46,0.957,30.0,16.5
1,Cepheid,"(15.0, 30.0]",19,1.000,42.0,20.0
2,Cepheid,"(30.0, 45.0]",39,1.000,52.0,22.0
3,Cepheid,"(45.0, 60.0]",66,0.970,53.5,29.0
4,Cepheid,"(60.0, 75.0]",25,1.000,45.0,28.0
5,Cepheid,"(75.0, 90.0]",305,0.997,37.0,26.0
6,Eclipsing Binary,"(-0.001, 15.0]",129,0.016,44.0,18.0
7,Eclipsing Binary,"(15.0, 30.0]",127,0.024,30.0,17.0
8,Eclipsing Binary,"(60.0, 75.0]",52,0.019,58.0,27.0
9,Eclipsing Binary,"(75.0, 90.0]",192,0.031,41.0,27.0


,class_name,epoch_bin,stars,recovery,median_latitude
0,Cepheid,"(19.999, 30.0]",54,0.963,18.718
1,Cepheid,"(30.0, 40.0]",229,0.996,84.551
2,Cepheid,"(40.0, 50.0]",115,0.991,65.008
3,Cepheid,"(50.0, 75.0]",70,0.986,49.140
4,Cepheid,"(75.0, 100.0]",26,1.000,83.899
5,Cepheid,"(100.0, inf]",6,1.000,45.380
6,Eclipsing Binary,"(19.999, 30.0]",76,0.039,28.537
7,Eclipsing Binary,"(30.0, 40.0]",177,0.017,75.965
8,Eclipsing Binary,"(40.0, 50.0]",104,0.029,8.972
9,Eclipsing Binary,"(50.0, 75.0]",133,0.023,74.295


## Period Coverage

In [33]:
cycle_bins = [0, 1, 2, 3, 4, 6, 10, 20, 50, np.inf]
cycle_data = period_validation.loc[period_validation["period_eligible"]].copy()
cycle_data["cycle_bin"] = pd.cut(
    cycle_data["n_cycles"], cycle_bins, include_lowest=True
)
cycle_recovery = (
    cycle_data.groupby(["class_name", "cycle_bin"], observed=True)
    .agg(
        stars=("source_id", "size"),
        recovery=("phase_coherent_recovery", "mean"),
        median_period=("gaia_period", "median"),
        median_epochs=("n_epochs", "median")
    ).reset_index()
)
display(cycle_recovery.round(3))

,class_name,cycle_bin,stars,recovery,median_period,median_epochs
0,Cepheid,"(6.0, 10.0]",1,1.000,108.654,82.0
1,Cepheid,"(10.0, 20.0]",3,1.000,74.463,59.0
2,Cepheid,"(20.0, 50.0]",16,0.938,26.152,37.5
3,Cepheid,"(50.0, inf]",480,0.992,3.069,39.0
4,Eclipsing Binary,"(4.0, 6.0]",2,0.000,189.313,60.0
5,Eclipsing Binary,"(10.0, 20.0]",3,0.000,68.273,62.0
6,Eclipsing Binary,"(20.0, 50.0]",8,0.000,29.984,36.0
7,Eclipsing Binary,"(50.0, inf]",487,0.025,0.412,40.0
8,Long-Period Variable,"(2.0, 3.0]",114,0.728,384.552,43.5
9,Long-Period Variable,"(3.0, 4.0]",47,0.830,268.432,43.0


## Classification Checks

In [35]:
if comparison["source_id"].duplicated().any() or oracle_data["source_id"].duplicated().any():
    raise ValueError("Duplicate classification sources.")

missing = set(comparison["source_id"]) - set(oracle_data["source_id"])
extra = set(oracle_data["source_id"]) - set(comparison["source_id"])

if missing or extra:
    raise ValueError(f"Oracle source mismatch: {len(missing)} missing, {len(extra)} extra.")

comparison = comparison.reset_index(drop=True)
oracle_data = oracle_data.set_index("source_id").loc[comparison["source_id"]].reset_index()

if not comparison["class_name"].equals(oracle_data["class_name"]):
    raise ValueError("Oracle and estimated class labels differ.")

if set(compare_cols) - set(oracle_data.columns):
    raise ValueError("Oracle comparison contains missing predictors.")

if not set(period_cols).isdisjoint(free_cols):
    raise ValueError("Period-dependent features entered the period-free experiment.")

if not set(color_cols).issubset(free_cols):
    raise ValueError("Color or amplitude is missing from period-free predictors.")

if not predictions["source_id"].isin(comparison["source_id"]).all():
    raise ValueError("Classification predictions contain unknown sources.")

if classification["accuracy"].isna().any():
    raise ValueError("Classification accuracy contains missing values.")

if not classification["accuracy"].between(0, 1).all():
    raise ValueError("Classification accuracy lies outside [0, 1].")

print("Classification sources:", len(comparison))
print("Wrong-period sources:", comparison["period_status"].ne("exact").sum())
print("Classification results:", len(classification))

Classification sources: 1879
Wrong-period sources: 711
Classification results: 270


## Export Results

In [36]:
FE_FEATURE_FILE.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(FE_FEATURE_FILE, index=False)
ablation.to_csv(FE_SAMPLING_FILE, index=False)
period_validation.to_csv(FE_VALIDATION_FILE, index=False)
oracle.to_csv(FE_ORACLE_FILE, index=False)
subsampling.to_csv(FE_SUBSAMPLE_FILE, index=False)
wrong_predictions.to_csv(FE_WRONG_FILE, index=False)
classification.to_csv(FE_RESULTS_FILE, index=False)
selected_features.to_csv(FE_CORR_FILE, index=False)

print("Features:", FE_FEATURE_FILE)
print("Sampling features:", FE_SAMPLING_FILE)
print("Period validation:", FE_VALIDATION_FILE)
print("Oracle:", FE_ORACLE_FILE)
print("Subsampling:", FE_SUBSAMPLE_FILE)
print("Wrong-period predictions:", FE_WRONG_FILE)
print("Classification:", FE_RESULTS_FILE)
print("Selected predictors:", FE_CORR_FILE)

Features: /Users/Sumaitat/Documents/Coding/gaia-varstars/data/processed/gaia_features_fe.csv
Sampling features: /Users/Sumaitat/Documents/Coding/gaia-varstars/data/processed/gaia_features_fe_sampling.csv
Period validation: /Users/Sumaitat/Documents/Coding/gaia-varstars/data/processed/gaia_period_validation_fe.csv
Oracle: /Users/Sumaitat/Documents/Coding/gaia-varstars/data/processed/gaia_features_fe_oracle.csv
Subsampling: /Users/Sumaitat/Documents/Coding/gaia-varstars/data/processed/gaia_fe_subsampling.csv
Wrong-period predictions: /Users/Sumaitat/Documents/Coding/gaia-varstars/data/processed/gaia_fe_wrong_period.csv
Classification: /Users/Sumaitat/Documents/Coding/gaia-varstars/data/processed/gaia_fe_classification.csv
Selected predictors: /Users/Sumaitat/Documents/Coding/gaia-varstars/data/processed/gaia_fe_selected.csv


## Final Checks

In [37]:
outputs = {
    "features": (FE_FEATURE_FILE, features),
    "sampling_features": (FE_SAMPLING_FILE, ablation),
    "validation": (FE_VALIDATION_FILE, period_validation),
    "oracle": (FE_ORACLE_FILE, oracle),
    "subsampling": (FE_SUBSAMPLE_FILE, subsampling),
    "wrong_period": (FE_WRONG_FILE, wrong_predictions),
    "classification": (FE_RESULTS_FILE, classification),
    "selected_features": (FE_CORR_FILE, selected_features)
}

for name, (path, original) in outputs.items():
    saved = pd.read_csv(path)

    if len(saved) != len(original):
        raise ValueError(f"{name}: saved row count differs.")

    if list(saved.columns) != list(original.columns):
        raise ValueError(f"{name}: saved columns differ.")

    print(f"{name}: {len(saved)} rows, {len(saved.columns)} columns")

if set(features["source_id"]) != set(ablation["source_id"]):
    raise ValueError("Feature and ablation source coverage differs.")

if set(period_validation["source_id"]) != set(sources["source_id"]):
    raise ValueError("Period validation source coverage is incomplete.")

if set(oracle["source_id"]) != set(stars.loc[stars["period_eligible"], "source_id"]):
    raise ValueError("Oracle source coverage is incomplete.")

print("Done.")

features: 1978 rows, 58 columns
sampling_features: 1978 rows, 67 columns
validation: 2000 rows, 45 columns
oracle: 1879 rows, 25 columns
subsampling: 46864 rows, 9 columns
wrong_period: 13104 rows, 9 columns
classification: 270 rows, 10 columns
selected_features: 56 rows, 2 columns
Done.
